In [1]:
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from category_encoders import LeaveOneOutEncoder
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv("datasets/crop_yield_dataset.csv")

# Rename columns for easier access
df = df.rename(columns={
    "Temperature (°C)": "temperature",
    "Rainfall (mm)": "rainfall",
    "Humidity (%)": "humidity",
    "Soil Type": "soil_type",
    "Weather Condition": "weather_condition",
    "Crop Type": "crop_type",
    "Yield (tons/hectare)": "yield"
})

In [3]:
X = df.drop("yield", axis=1)
y = df["yield"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
X_train_OHE = pd.get_dummies(X_train, columns=["soil_type", "weather_condition", "crop_type"], drop_first=True)
X_test_OHE = pd.get_dummies(X_test, columns=["soil_type", "weather_condition", "crop_type"], drop_first=True)

In [6]:
LOOE = LeaveOneOutEncoder(cols=["soil_type", "weather_condition", "crop_type"])
X_train_LOOE = LOOE.fit_transform(X_train, y_train)
X_test_LOOE = LOOE.transform(X_test)

In [7]:
scaler_std = StandardScaler()
numeric_columns = ["temperature", "rainfall", "humidity"]

In [8]:
X_train_OHE_scaled_std = scaler_std.fit_transform(X_train_OHE[numeric_columns])
X_test_OHE_scaled_std = scaler_std.transform(X_test_OHE[numeric_columns])
final_train_OHE_scaled = pd.concat([pd.DataFrame(X_train_OHE_scaled_std, columns=numeric_columns, index=X_train_OHE.index), X_train_OHE.drop(columns=numeric_columns)], axis=1)
final_test_OHE_scaled = pd.concat([pd.DataFrame(X_test_OHE_scaled_std, columns=numeric_columns, index=X_test_OHE.index), X_test_OHE.drop(columns=numeric_columns)], axis=1)


In [9]:
X_train_LOOE_scaled_std = scaler_std.fit_transform(X_train_LOOE[numeric_columns])
X_test_LOOE_scaled_std = scaler_std.transform(X_test_LOOE[numeric_columns])
final_train_LOOE_scaled = pd.concat([pd.DataFrame(X_train_LOOE_scaled_std, columns=numeric_columns, index=X_train_LOOE.index), X_train_LOOE.drop(columns=numeric_columns)], axis=1)
final_test_LOOE_scaled = pd.concat([pd.DataFrame(X_test_LOOE_scaled_std, columns=numeric_columns, index=X_test_LOOE.index), X_test_LOOE.drop(columns=numeric_columns)], axis=1)

In [10]:
scaler_MM = MinMaxScaler()
X_train_OHE_scaled_MM = scaler_MM.fit_transform(X_train_OHE[numeric_columns])
X_test_OHE_scaled_MM = scaler_MM.transform(X_test_OHE[numeric_columns])
final_train_OHE_scaled_MM = pd.concat([pd.DataFrame(X_train_OHE_scaled_MM, columns=numeric_columns, index=X_train_OHE.index), X_train_OHE.drop(columns=numeric_columns)], axis=1)
final_test_OHE_scaled_MM = pd.concat([pd.DataFrame(X_test_OHE_scaled_MM, columns=numeric_columns, index=X_test_OHE.index), X_test_OHE.drop(columns=numeric_columns)], axis=1)

In [11]:
X_train_LOOE_scaled_MM = scaler_MM.fit_transform(X_train_LOOE[numeric_columns])
X_test_LOOE_scaled_MM = scaler_MM.transform(X_test_LOOE[numeric_columns])
final_train_LOOE_scaled_MM = pd.concat([pd.DataFrame(X_train_LOOE_scaled_MM, columns=numeric_columns, index=X_train_LOOE.index), X_train_LOOE.drop(columns=numeric_columns)], axis=1)
final_test_LOOE_scaled_MM = pd.concat([pd.DataFrame(X_test_LOOE_scaled_MM, columns=numeric_columns, index=X_test_LOOE.index), X_test_LOOE.drop(columns=numeric_columns)], axis=1)


In [12]:
datasets = {
    "OHE + StandardScaler": (final_train_OHE_scaled, final_test_OHE_scaled),
    "OHE + MinMaxScaler": (final_train_OHE_scaled_MM, final_test_OHE_scaled_MM),
    "LOOE + StandardScaler": (final_train_LOOE_scaled, final_test_LOOE_scaled),
    "LOOE + MinMaxScaler": (final_train_LOOE_scaled_MM, final_test_LOOE_scaled_MM)
}

In [13]:
models = {
    "Linear Regression": LinearRegression(),
    "K-Nearest Neighbors": KNeighborsRegressor(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Support Vector Regressor (SVR)": SVR()
}

In [14]:
for data_name, (X_train_df, X_test_df) in datasets.items():
    print(f"Dataset: {data_name}")
    print("-" * (len(data_name) + 10))
    
    try:
        clean_train_indices = X_train_df.replace([np.inf, -np.inf], np.nan).dropna().index
        
        X_train_clean = X_train_df.loc[clean_train_indices].drop(columns='split', errors='ignore')
        y_train_clean = y_train.loc[clean_train_indices]

        clean_test_indices = X_test_df.replace([np.inf, -np.inf], np.nan).dropna().index
        X_test_clean = X_test_df.loc[clean_test_indices].drop(columns='split', errors='ignore')
        y_test_clean = y_test.loc[clean_test_indices]
            
        for model_name, model in models.items():
            model.fit(X_train_clean, y_train_clean)
            y_preds = model.predict(X_test_clean)
            mse = mean_squared_error(y_test_clean, y_preds)
            r2 = r2_score(y_test_clean, y_preds)
            print(f"  -> {model_name}:")
            print(f"     - MSE: {mse:.4f}")
            print(f"     - R2: {r2:.4f}")
        print("\n" + "="*50 + "\n")
    except Exception as e:
        print(f"An error occurred while processing {data_name} dataset: {e}")
        print("\n" + "="*50 + "\n")

Dataset: OHE + StandardScaler
------------------------------
  -> Linear Regression:
     - MSE: 5.2129
     - R2: -0.0306
  -> K-Nearest Neighbors:
     - MSE: 5.7518
     - R2: -0.1372
  -> Random Forest:
     - MSE: 5.1911
     - R2: -0.0263
  -> Support Vector Regressor (SVR):
     - MSE: 5.4424
     - R2: -0.0760


Dataset: OHE + MinMaxScaler
----------------------------
  -> Linear Regression:
     - MSE: 5.2129
     - R2: -0.0306
  -> K-Nearest Neighbors:
     - MSE: 6.5172
     - R2: -0.2885
  -> Random Forest:
     - MSE: 5.1911
     - R2: -0.0263
  -> Support Vector Regressor (SVR):
     - MSE: 5.6576
     - R2: -0.1185


Dataset: LOOE + StandardScaler
-------------------------------
  -> Linear Regression:
     - MSE: 4.9845
     - R2: 0.0145
  -> K-Nearest Neighbors:
     - MSE: 6.1415
     - R2: -0.2142
  -> Random Forest:
     - MSE: 5.1499
     - R2: -0.0182
  -> Support Vector Regressor (SVR):
     - MSE: 5.0763
     - R2: -0.0036


Dataset: LOOE + MinMaxScaler
--------

In [15]:
sample=pd.DataFrame([{
  "temperature":30,
  "rainfall":550,
  "humidity":75,
  "soil_type":"Sandy",
  "weather_condition":"Sunny",
  "crop_type":"Corn"
}])

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [17]:
categorical_features = ["soil_type", "weather_condition", "crop_type"]
numeric_features = ["temperature", "rainfall","humidity"]

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical_encoding", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), categorical_features),
        ("numerical_scaling", StandardScaler(), numeric_features)
    ],
    remainder='passthrough'
)

In [19]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

In [20]:
# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Export the pipeline using pickle
with open("pipeline.pkl", "wb") as f:
    pickle.dump(pipeline, f)

# Load the pipeline back (optional, for demonstration)
with open("pipeline.pkl", "rb") as f:
    loaded_pipeline = pickle.load(f)

# Predict using the loaded pipeline for the sample data
prediction = loaded_pipeline.predict(sample)
print(f"The predicted yield for your sample data is: {prediction[0]:.4f} tons/hectare")

The predicted yield for your sample data is: 5.6058 tons/hectare


In [21]:
loaded_pipeline

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categorical_encoding', ...), ('numerical_scaling', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [22]:
X_train_LOOE_scaled_std = scaler_std.fit_transform(X_train_LOOE[numeric_columns])
X_test_LOOE_scaled_std = scaler_std.transform(X_test_LOOE[numeric_columns])

final_train_LOOE_scaled = pd.concat([pd.DataFrame(X_train_LOOE_scaled_std, columns=numeric_columns, index=X_train_LOOE.index), X_train_LOOE.drop(columns=numeric_columns)], axis=1)
final_test_LOOE_scaled = pd.concat([pd.DataFrame(X_test_LOOE_scaled_std, columns=numeric_columns, index=X_test_LOOE.index), X_test_LOOE.drop(columns=numeric_columns)], axis=1)

model = LinearRegression()

clean_train_indices = final_train_LOOE_scaled.replace([np.inf, -np.inf], np.nan).dropna().index
X_train_clean = final_train_LOOE_scaled.loc[clean_train_indices]
y_train_clean = y_train.loc[clean_train_indices]

clean_test_indices = final_test_LOOE_scaled.replace([np.inf, -np.inf], np.nan).dropna().index
X_test_clean = final_test_LOOE_scaled.loc[clean_test_indices]
y_test_clean = y_test.loc[clean_test_indices]

model.fit(X_train_clean, y_train_clean)
predictions = model.predict(X_test_clean)

print("Predicted yields for every row in the LOOE + StandardScaler test dataset:")
print(predictions)

results_df = X_test.loc[clean_test_indices].copy()
results_df['predicted_yield'] = predictions
results_df['actual_yield'] = y_test_clean
print("\nOriginal test data with predicted and actual yields:")
print(results_df.head())

Predicted yields for every row in the LOOE + StandardScaler test dataset:
[5.77285093 6.27449929 5.57456457 5.93979484 5.74149952 5.69437141
 5.67980684 5.69709456 6.05191031 5.802565   6.26299096 5.41369944
 6.02082927 5.77770352 6.17903545 6.02228094 5.76942212 5.75699948
 5.87346158 5.90684354 5.69865462 5.95712299 6.37593646 5.99595742
 6.01117628 5.79248445 6.25176596 5.63871113 5.81178479 5.90127915
 5.78158767 6.00563242 5.76929727 5.9839993  6.06988407 5.81845535
 6.3217029  5.86340873 6.02721724 5.79537989 5.66065681 6.2187254
 6.14599267 5.82292741 6.10099101 5.88147922 5.97082686 6.15167636
 5.8178737  5.82898518 5.78810102 6.03474286 5.69131653 5.91886253
 5.5803782  5.53381066 5.92561383 5.89342121 5.77923406 5.99156782
 5.69326316 6.09520563 6.05708177 6.15547514 6.1909439  5.51315093
 6.21438547 6.04146074 6.30338418 5.9067588  5.69746531 6.37041351
 6.03896031 6.25001129 6.18322836 5.77186773 5.74448047 5.89545207
 5.97542047 5.91814196 5.93254389 5.74052042 5.88754267 

In [23]:
import pandas as pd

In [25]:
from pathlib import Path
import joblib

# 👇 Replace 'pipeline' with the actual variable that holds your trained model
# e.g., 'model', 'rf', 'regressor', etc.
trained_model = pipeline  

# Path to fastapi/models folder
pipeline_path = Path("../fastapi/models/pipeline.pkl")
pipeline_path.parent.mkdir(parents=True, exist_ok=True)

# Save
joblib.dump(trained_model, pipeline_path)
print(f"✅ Pipeline saved at {pipeline_path.resolve()}")


✅ Pipeline saved at D:\Agri_Yield\fastapi\models\pipeline.pkl


In [26]:
import joblib
pipeline = joblib.load("../fastapi/models/pipeline.pkl")
print("✅ Loaded pipeline:", type(pipeline))


✅ Loaded pipeline: <class 'sklearn.pipeline.Pipeline'>


In [24]:
final_pipeline = pd.read_csv("D:/Agri_Yield/fastpi/final_pipeline.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'D:/Agri_Yield/fastpi/final_pipeline.csv'